# 03 · Queries de verificación — respondiendo al negocio

Seis queries analíticas que demuestran que el modelo soporta las necesidades que planteaba
el enunciado. Cada una responde a una pregunta concreta de negocio y **usa una construcción
SQL distinta**, para cubrir todo el temario del sprint.

| # | Pregunta de negocio | SQL que ejercita |
|---|---|---|
| 1 | ¿Cómo evolucionan los ingresos mes a mes? | Agregación temporal con `GROUP BY` sobre fecha |
| 2 | ¿Qué productos venden más y cuál es su margen? | `JOIN` + cálculo de margen con `price` y `cost` |
| 3 | ¿Qué rentabilidad tiene cada categoría? | Agregación por categoría y ordenación |
| 4 | ¿Cuánto vale un cliente según su país y canal? | `GROUP BY` de dos dimensiones |
| 5 | ¿Cuánto se tarda en entregar? | Diferencia de fechas y estadísticos |
| 6 | ¿Qué productos tienen peor valoración? | `JOIN` de 4 tablas + `HAVING` |

> **Nota sobre el dinero.** El ingreso real de una línea es
> `unit_price * quantity * (1 - discount_pct/100)`, y solo cuenta si el pago está
> `completed`: un pedido reembolsado o fallido **no es ingreso**. Esa condición se repite en
> todas las queries de facturación, y es justo el tipo de matiz que un modelo mal diseñado
> haría imposible de expresar.

## Conexión

In [1]:
import os
import sys

# el proyecto se importa desde la raiz del repo, no desde notebooks/
sys.path.insert(0, os.path.abspath(".."))

import pandas as pd
from dotenv import load_dotenv

load_dotenv(os.path.join("..", ".env"))   # si no existe, no pasa nada
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

from src.esquema import ESQUEMA, ORDEN_CARGA, campos, ddl_sqlite, foraneas, resumen
from src.conexion import obtener_backend

backend = obtener_backend()

# atajos para que las queries se lean bien
CUSTOMERS  = backend.tabla("customers")
CATEGORIES = backend.tabla("categories")
PRODUCTS   = backend.tabla("products")
ORDERS     = backend.tabla("orders")
ITEMS      = backend.tabla("order_items")
PAYMENTS   = backend.tabla("payments")
REVIEWS    = backend.tabla("reviews")

print("\nConectado a:", backend.nombre)

Backend: SQLite (espejo local). Motivo: GCP_PROJECT_ID no está configurado en el .env.
Las queries son SQL estándar: el resultado es equivalente al de BigQuery.

Conectado a: SQLite (espejo local)


## Query 1 — Evolución mensual de los ingresos

`SUBSTR(ordered_at, 1, 7)` extrae el `YYYY-MM` de la fecha. En BigQuery con una columna
`DATE` real se usaría `FORMAT_DATE('%Y-%m', o.ordered_at)`; con el espejo local, donde las
fechas van como texto ISO, `SUBSTR` da el mismo resultado y funciona en ambos.

In [2]:
q1 = backend.query(f"""
SELECT SUBSTR(o.ordered_at, 1, 7)                                   AS mes,
       COUNT(DISTINCT o.order_id)                                   AS pedidos,
       ROUND(SUM(oi.unit_price * oi.quantity * (1 - oi.discount_pct / 100)), 2) AS ingresos,
       ROUND(SUM(oi.unit_price * oi.quantity * (1 - oi.discount_pct / 100))
             / COUNT(DISTINCT o.order_id), 2)                       AS ticket_medio
FROM {ITEMS} oi
JOIN {ORDERS}   o ON o.order_id = oi.order_id
JOIN {PAYMENTS} pa ON pa.order_id = o.order_id
WHERE pa.payment_status = 'completed'
GROUP BY mes
ORDER BY mes
""")

print(f"{len(q1)} meses con facturación")
display(q1.head(12))
print(f"Ingresos totales del periodo: {q1['ingresos'].sum():,.2f} EUR")

24 meses con facturación


,mes,pedidos,ingresos,ticket_medio
0,2024-01,1,2944.16,2944.16
1,2024-02,3,5635.68,1878.56
2,2024-03,5,7463.54,1492.71
3,2024-04,10,26343.42,2634.34
4,2024-05,14,31255.84,2232.56
5,2024-06,15,35328.78,2355.25
6,2024-07,21,41840.55,1992.41
7,2024-08,27,57633.67,2134.58
8,2024-09,33,72973.62,2211.32
9,2024-10,30,54623.19,1820.77


Ingresos totales del periodo: 3,920,527.88 EUR


## Query 2 — Top 10 de productos por ingresos

Aquí se ve por qué `products` necesita guardar **`cost`** además de `price`: sin el coste no
hay forma de calcular el margen, y sin margen no se sabe si el producto que más factura es
además el que más deja.

In [3]:
q2 = backend.query(f"""
SELECT p.product_name,
       c.category_name,
       SUM(oi.quantity)                                             AS unidades,
       ROUND(SUM(oi.unit_price * oi.quantity * (1 - oi.discount_pct / 100)), 2) AS ingresos,
       ROUND(SUM((oi.unit_price * (1 - oi.discount_pct / 100) - p.cost) * oi.quantity), 2) AS margen,
       ROUND(100.0 * SUM((oi.unit_price * (1 - oi.discount_pct / 100) - p.cost) * oi.quantity)
             / SUM(oi.unit_price * oi.quantity * (1 - oi.discount_pct / 100)), 1) AS margen_pct
FROM {ITEMS} oi
JOIN {PRODUCTS}   p ON p.product_id = oi.product_id
JOIN {CATEGORIES} c ON c.category_id = p.category_id
JOIN {ORDERS}     o ON o.order_id = oi.order_id
JOIN {PAYMENTS}  pa ON pa.order_id = o.order_id
WHERE pa.payment_status = 'completed'
GROUP BY p.product_id, p.product_name, c.category_name
ORDER BY ingresos DESC
LIMIT 10
""")

display(q2)

,product_name,category_name,unidades,ingresos,margen,margen_pct
0,Orbit Ultra 4,Laptops,77,180833.87,26574.38,14.7
1,Lumix Neo 10,Laptops,79,171498.00,13087.20,7.6
2,Krono One 4,Laptops,87,168784.21,13544.02,8.0
3,Orbit Neo 8,Laptops,82,168345.30,17206.18,10.2
4,Voltia Pro 14,Laptops,83,155297.06,11480.47,7.4
5,Voltia Lite 7,Laptops,70,149778.47,9045.57,6.0
6,Krono Mini 6,Laptops,74,140253.69,13149.07,9.4
7,Lumix Neo 9,Laptops,94,126970.36,17955.74,14.1
8,Lumix Plus 4,Tablets,85,113436.27,14728.32,13.0
9,Orbit Lite 3,Smartphones,79,101377.47,16806.39,16.6


## Query 3 — Rentabilidad por categoría

La pregunta que de verdad importa a dirección: no qué categoría **factura** más, sino cuál
**deja** más. No siempre coinciden.

In [4]:
q3 = backend.query(f"""
SELECT c.category_name,
       COUNT(DISTINCT p.product_id)                                 AS productos,
       SUM(oi.quantity)                                             AS unidades,
       ROUND(SUM(oi.unit_price * oi.quantity * (1 - oi.discount_pct / 100)), 2) AS ingresos,
       ROUND(SUM((oi.unit_price * (1 - oi.discount_pct / 100) - p.cost) * oi.quantity), 2) AS margen,
       ROUND(100.0 * SUM((oi.unit_price * (1 - oi.discount_pct / 100) - p.cost) * oi.quantity)
             / SUM(oi.unit_price * oi.quantity * (1 - oi.discount_pct / 100)), 1) AS margen_pct
FROM {ITEMS} oi
JOIN {PRODUCTS}   p ON p.product_id = oi.product_id
JOIN {CATEGORIES} c ON c.category_id = p.category_id
JOIN {ORDERS}     o ON o.order_id = oi.order_id
JOIN {PAYMENTS}  pa ON pa.order_id = o.order_id
WHERE pa.payment_status = 'completed'
GROUP BY c.category_id, c.category_name
ORDER BY margen DESC
""")

display(q3)
print("Categoría que más factura:", q3.sort_values("ingresos", ascending=False).iloc[0]["category_name"])
print("Categoría que más margen deja:", q3.iloc[0]["category_name"])

,category_name,productos,unidades,ingresos,margen,margen_pct
0,Wearables,17,1391,620319.50,165169.32,26.6
1,Laptops,15,1170,1694536.74,156701.29,9.2
2,Smartphones,9,703,583651.57,73598.05,12.6
3,Periféricos,8,626,244449.79,70961.44,29.0
4,Tablets,7,544,374410.87,57701.48,15.4
5,Smart Home,6,485,133708.47,46097.16,34.5
6,Audio,5,407,112407.55,33857.40,30.1
7,Gaming,3,254,157043.38,25846.95,16.5


Categoría que más factura: Laptops
Categoría que más margen deja: Wearables


## Query 4 — Valor del cliente por país y canal de captación

Cruza dos dimensiones para responder a la pregunta de marketing: ¿qué combinación de país y
canal trae a los clientes que más gastan? Filtro a los grupos con al menos 10 clientes para
no sacar conclusiones de una muestra de tres personas.

In [5]:
q4 = backend.query(f"""
SELECT cu.country,
       cu.acquisition_channel                                       AS canal,
       COUNT(DISTINCT cu.customer_id)                               AS clientes,
       COUNT(DISTINCT o.order_id)                                   AS pedidos,
       ROUND(SUM(oi.unit_price * oi.quantity * (1 - oi.discount_pct / 100)), 2) AS ingresos,
       ROUND(SUM(oi.unit_price * oi.quantity * (1 - oi.discount_pct / 100))
             / COUNT(DISTINCT cu.customer_id), 2)                   AS gasto_por_cliente
FROM {CUSTOMERS} cu
JOIN {ORDERS}    o  ON o.customer_id = cu.customer_id
JOIN {ITEMS}     oi ON oi.order_id = o.order_id
JOIN {PAYMENTS}  pa ON pa.order_id = o.order_id
WHERE pa.payment_status = 'completed'
GROUP BY cu.country, cu.acquisition_channel
HAVING COUNT(DISTINCT cu.customer_id) >= 10
ORDER BY gasto_por_cliente DESC
LIMIT 12
""")

display(q4)

,country,canal,clientes,pedidos,ingresos,gasto_por_cliente
0,Alemania,social,16,59,160008.38,10000.52
1,España,referral,14,55,128305.89,9164.71
2,Alemania,organic,23,88,208480.60,9064.37
3,España,social,31,109,277027.33,8936.37
4,Italia,organic,17,60,150759.06,8868.18
5,Alemania,email,11,37,97446.77,8858.80
6,España,organic,59,205,511263.65,8665.49
7,Alemania,paid_ads,11,40,93025.37,8456.85
8,España,email,28,95,220936.96,7890.61
9,Italia,social,11,35,86010.69,7819.15


## Query 5 — Tiempo de entrega por país

Métrica operativa. `JULIANDAY` es la función de SQLite para restar fechas; en BigQuery el
equivalente es `DATE_DIFF(o.delivered_at, o.ordered_at, DAY)`. Solo cuentan los pedidos
efectivamente entregados: el resto no tiene tiempo de entrega que medir.

In [6]:
funcion_dias = ("DATE_DIFF(o.delivered_at, o.ordered_at, DAY)" if backend.es_bigquery
                else "JULIANDAY(o.delivered_at) - JULIANDAY(o.ordered_at)")

q5 = backend.query(f"""
SELECT o.shipping_country                       AS pais,
       COUNT(*)                                 AS entregas,
       ROUND(AVG({funcion_dias}), 2)            AS dias_medios,
       MIN({funcion_dias})                      AS dias_min,
       MAX({funcion_dias})                      AS dias_max
FROM {ORDERS} o
WHERE o.status = 'delivered'
  AND o.delivered_at IS NOT NULL
GROUP BY o.shipping_country
ORDER BY dias_medios
""")

display(q5)

,pais,entregas,dias_medios,dias_min,dias_max
0,Bélgica,51,5.92,2.0,11.0
1,Francia,159,6.33,2.0,11.0
2,España,501,6.54,2.0,11.0
3,Alemania,209,6.61,2.0,11.0
4,Países Bajos,62,6.71,2.0,10.0
5,Portugal,130,6.72,2.0,11.0
6,Italia,156,6.92,2.0,11.0


## Query 6 — Productos peor valorados

Encadena las cuatro tablas del camino `reviews → order_items → products → categories`.
Aquí se ve la razón de que `reviews` cuelgue de `order_items` y no de `orders`: si colgara
del pedido, constaría que un cliente quedó descontento pero **no con qué producto**, y esta
query sería imposible de escribir.

El `HAVING COUNT(*) >= 5` evita que un producto con una única reseña de 1 estrella encabece
la lista.

In [7]:
q6 = backend.query(f"""
SELECT p.product_name,
       c.category_name,
       COUNT(*)                     AS n_valoraciones,
       ROUND(AVG(r.rating), 2)      AS nota_media,
       SUM(CASE WHEN r.rating <= 2 THEN 1 ELSE 0 END) AS valoraciones_negativas
FROM {REVIEWS}    r
JOIN {ITEMS}      oi ON oi.order_item_id = r.order_item_id
JOIN {PRODUCTS}   p  ON p.product_id = oi.product_id
JOIN {CATEGORIES} c  ON c.category_id = p.category_id
GROUP BY p.product_id, p.product_name, c.category_name
HAVING COUNT(*) >= 5
ORDER BY nota_media ASC
LIMIT 10
""")

display(q6)

media_global = backend.query(f"SELECT ROUND(AVG(rating), 2) AS media FROM {REVIEWS}")["media"][0]
print(f"Nota media global del catálogo: {media_global}")

,product_name,category_name,n_valoraciones,nota_media,valoraciones_negativas
0,Nexor Ultra 3,Periféricos,18,3.28,5
1,Zentra Mini 14,Smart Home,17,3.47,4
2,Voltia Pro 14,Laptops,16,3.50,5
3,Lumix Edge 3,Tablets,19,3.53,3
4,Nexor Mini 1,Wearables,14,3.57,4
5,Krono Mini 5,Smartphones,22,3.59,4
6,Zentra Neo 1,Wearables,20,3.65,3
7,Voltia Lite 13,Tablets,12,3.67,2
8,Aurio Max 10,Audio,13,3.69,2
9,Lumix Plus 8,Wearables,13,3.77,2


Nota media global del catálogo: 4.07


---

## Conclusiones

**El modelo soporta las seis preguntas de negocio del enunciado**, y cada una necesita una
pieza concreta del diseño:

| La query necesita... | Porque el modelo tiene... |
|---|---|
| Separar ingresos reales de pedidos reembolsados | `payments.payment_status` en tabla aparte |
| Calcular margen | `products.cost` junto a `products.price` |
| Valorar pedidos históricos con su precio real | `order_items.unit_price` y no `products.price` |
| Saber qué producto concreto decepcionó | `reviews` colgando de `order_items`, no de `orders` |
| Medir tiempos de entrega | `ordered_at` / `shipped_at` / `delivered_at` separadas y nullables |
| Segmentar por origen del cliente | `country` y `acquisition_channel` en `customers` |

Cada una de esas decisiones se justifica en `docs/normalizacion.md`.

### Portabilidad BigQuery / SQLite

Las seis queries son SQL estándar y corren igual en los dos backends. Solo hubo que adaptar
una cosa: **la aritmética de fechas**, que no está estandarizada
(`DATE_DIFF` en BigQuery, `JULIANDAY` en SQLite). Se resuelve en la query 5 eligiendo la
función según el backend activo.

In [8]:
backend.cerrar()
print("Verificación completada.")

Verificación completada.
